<br/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="left"/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="right"/>
<div align="center">
<h2>Bootcamp Data Science — Modulo 2</h2><br/>
<h1>Semana 8 · Martes — LightGBM</h1>
<h3>El gradient boosting de Microsoft, rapido y potente</h3>
<br/>
    <b>Instructor:</b> Jesus Ortiz · jesus.jeduardo7@gmail.com<br/><br/>
    <b>SkillNest · b2b-sonda-data-science</b>
</div>
<br/>

## Objetivos

Al final de la clase van a poder:

1. Entender que es el gradient boosting y por que LightGBM lo hace mejor que sklearn.
2. Diferenciar LightGBM de Random Forest (a pesar de que ambos usan arboles).
3. Dominar los hiperparametros clave: num_leaves, learning_rate, n_estimators, max_depth.
4. Usar early stopping para que el modelo se detenga solo.
5. Combinar LightGBM con Optuna (lo de ayer) para un tuning profesional.
6. Resolver UN ejercicio largo donde LightGBM compita contra Random Forest y Logistic Regression en un dataset NUEVO.

# 1. Boosting vs Bagging

Hasta aca conociamos los ensembles tipo **Bagging** (Random Forest): muchos arboles entrenados en paralelo sobre muestras aleatorias, despues se promedia. Cada arbol es independiente.

**Boosting** es distinto: los arboles se entrenan **en secuencia**, donde cada arbol nuevo trata de corregir los errores del anterior. La analogia: es como tener un equipo de estudiantes resolviendo un examen donde cada uno se enfoca en las preguntas que los anteriores fallaron.

| | Bagging (RF) | Boosting (LightGBM, XGBoost) |
|---|---|---|
| Como entrena | Arboles en paralelo | Arboles en secuencia |
| Cada arbol corrige... | Nada, son independientes | Errores del arbol anterior |
| Velocidad | Rapido (paraleliza facil) | Mas lento (secuencial) |
| Overfitting | Mas resistente | Mas sensible, hay que cuidar |
| Suele ganar en... | Datasets simples | Competencias Kaggle, datasets tabulares grandes |

Si hay un modelo que gana Kaggle en problemas tabulares, es alguna variante de boosting: XGBoost, LightGBM o CatBoost. Por eso vale la pena conocerlos.

# 2. Por que LightGBM es "light"

Microsoft saco LightGBM en 2017 con dos optimizaciones clave que lo hacen entre 10 y 20 veces mas rapido que XGBoost:

1. **Histogram-based**: en vez de probar cada valor unico de cada feature, agrupa valores en bins (256 por defecto). Pierde precision teorica pero acelera muchisimo y casi no afecta el resultado.

2. **Leaf-wise growth**: los otros boosters crecen el arbol nivel por nivel (level-wise). LightGBM crece por hoja: en cada paso elige la hoja que reducira mas el error. Es mas eficiente.

El resultado: mismo rendimiento que XGBoost pero MUCHO mas rapido. En la practica, si tu dataset tiene mas de 10k filas, usa LightGBM.

Instalacion: `pip install lightgbm`

# 3. Hiperparametros clave

LightGBM tiene como 50 hiperparametros, pero estos son los que importan en el 90% de los casos:

| Hiperparametro | Que hace | Default | Rango tipico |
|---|---|---|---|
| `n_estimators` | Numero de arboles | 100 | 100-1000 |
| `learning_rate` | Cuanto contribuye cada arbol nuevo | 0.1 | 0.01-0.3 |
| `num_leaves` | Hojas maximas por arbol | 31 | 15-127 |
| `max_depth` | Profundidad maxima (-1 = sin limite) | -1 | 3-12 |
| `min_child_samples` | Muestras minimas por hoja | 20 | 5-100 |
| `reg_alpha` | Regularizacion L1 | 0 | 0-10 |
| `reg_lambda` | Regularizacion L2 | 0 | 0-10 |
| `subsample` | % de filas a usar por arbol | 1.0 | 0.6-1.0 |
| `colsample_bytree` | % de features a usar por arbol | 1.0 | 0.6-1.0 |

La regla mas importante: si subis `n_estimators`, baja `learning_rate` (y viceversa). Mas arboles con paso chico es mejor que pocos arboles con paso grande.

## Setup

Hoy vamos a usar un dataset que NUNCA hemos visto: **Adult Income (Census)**. Es un clasico de ML tabular, viene del censo de USA del 92 con datos demograficos y laborales. Queremos predecir si una persona gana mas o menos de 50K USD al ano.

Tiene 48,842 filas, 14 features (mezcla de numericas y categoricas) y dos clases (<=50K y >50K) con desbalance moderado (~3:1). Es el escenario perfecto para LightGBM porque maneja categoricas nativamente.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

import lightgbm as lgb

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

# Adult Income (Census). Primera vez tarda ~10s en bajar, despues queda en cache.
adult = fetch_openml('adult', version=2, as_frame=True, parser='auto')
X = adult.data.copy()
y = (adult.target == '>50K').astype(int)  # 1 = gana mas de 50K

# Convertimos categoricas a codes para que todos los modelos funcionen
cat_cols = X.select_dtypes(include='category').columns.tolist()
for c in cat_cols:
    X[c] = X[c].cat.codes

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Features categoricas: {len(cat_cols)} | numericas: {X.shape[1]-len(cat_cols)}')
print(f'Distribucion target: {y.value_counts(normalize=True).round(3).to_dict()}')

# 4. LightGBM basico

Primer entrenamiento con defaults razonables. Notese que es clasificacion binaria, asi que usamos `LGBMClassifier` y la metrica de referencia sera el AUC-ROC (robusto al desbalance) ademas de Accuracy y F1.

In [ ]:
modelo_lgb = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbose=-1
)

t0 = time.time()
modelo_lgb.fit(X_train, y_train)
t_lgb = time.time() - t0

pred = modelo_lgb.predict(X_test)
proba = modelo_lgb.predict_proba(X_test)[:, 1]

print(f'Accuracy: {accuracy_score(y_test, pred):.4f}')
print(f'F1:       {f1_score(y_test, pred):.4f}')
print(f'AUC-ROC:  {roc_auc_score(y_test, proba):.4f}')
print(f'Tiempo:   {t_lgb:.2f}s')

# 5. Early stopping: que el modelo decida cuantos arboles necesita

Una de las ventajas de LightGBM es el **early stopping**. Le decimos: entrena hasta n_estimators, pero si despues de M rondas no mejoras en validation, parate.

Esto:
- Evita overfitting (no sigues entrenando arboles cuando ya no mejoras).
- Ahorra tiempo (no entrenas arboles que sobran).
- Es basicamente magia.

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

modelo_es = lgb.LGBMClassifier(
    n_estimators=2000,  # Le decimos que puede entrenar hasta 2000
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbose=-1
)
modelo_es.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[lgb.early_stopping(stopping_rounds=50)]
)

pred_es = modelo_es.predict(X_test)
proba_es = modelo_es.predict_proba(X_test)[:, 1]

print(f'\nArboles que realmente entreno (best_iteration): {modelo_es.best_iteration_}')
print(f'Accuracy test: {accuracy_score(y_test, pred_es):.4f}')
print(f'AUC test:      {roc_auc_score(y_test, proba_es):.4f}')
print(f'\nVean que aunque le dije 2000, paro antes porque dejo de mejorar en validation.')

# 6. Feature Importance

Igual que Random Forest, LightGBM da importancia de features. Pero ofrece DOS tipos:

- `importance_type='gain'`: cuanto reduce el error cada feature (la mejor para interpretar).
- `importance_type='split'`: cuantas veces se uso cada feature (la mas facil de hacer pero menos informativa).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
lgb.plot_importance(modelo_lgb, max_num_features=10, importance_type='gain',
                     ax=axes[0], title='Importance por GAIN (cuanto reduce el error)')
lgb.plot_importance(modelo_lgb, max_num_features=10, importance_type='split',
                     ax=axes[1], title='Importance por SPLIT (cuantas veces se uso)')
plt.tight_layout(); plt.show()

# 7. Tuning de LightGBM con Optuna (lo de ayer)

Combinamos lo de ayer con lo de hoy. Optuna busca los mejores hiperparametros de LightGBM de forma inteligente.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves':       trial.suggest_int('num_leaves', 15, 127),
        'max_depth':        trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0, 10),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0, 10),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
    }
    modelo = lgb.LGBMClassifier(**params, random_state=42, verbose=-1)
    return cross_val_score(modelo, X_train, y_train, cv=3, scoring='roc_auc', n_jobs=-1).mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30, show_progress_bar=False)

print(f'Mejor AUC CV: {study.best_value:.4f}')
print(f'Mejores params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

modelo_tuneado = lgb.LGBMClassifier(**study.best_params, random_state=42, verbose=-1).fit(X_train, y_train)
proba_t = modelo_tuneado.predict_proba(X_test)[:, 1]
pred_t = modelo_tuneado.predict(X_test)
print(f'\nAccuracy test: {accuracy_score(y_test, pred_t):.4f}')
print(f'F1 test:       {f1_score(y_test, pred_t):.4f}')
print(f'AUC test:      {roc_auc_score(y_test, proba_t):.4f}')

---
# Ejercicio integrador

Un solo ejercicio, pero exigente. Comparen LightGBM contra los modelos que ya conocen.

## Reto: predecir si una persona gana mas de 50K USD/ano comparando 4 enfoques

Dataset: el mismo que vimos en clase (`Adult Income` via `fetch_openml`). Si no lo tienen cargado, usen el codigo del setup.

Compitiendo: Logistic Regression, RandomForest, LightGBM basico, LightGBM tuneado con Optuna.

### Lo que tienen que hacer

**Parte A — 3 baselines (15 min)**
1. Entrenar 3 modelos con parametros por defecto: LogisticRegression (con StandardScaler en un Pipeline), RandomForestClassifier, LGBMClassifier.
2. Reportar Accuracy, F1, AUC-ROC y tiempo de entrenamiento para cada uno.
3. Hacer una tabla comparativa.

**Parte B — LightGBM con early stopping (10 min)**
4. Hacer un split adicional train/val (stratify=y_train).
5. Entrenar LightGBM con n_estimators=2000 y early_stopping_rounds=50, eval_metric='auc'.
6. Reportar best_iteration y metricas en test.
7. Cuantos arboles entreno al final? Mejoro respecto al baseline de LightGBM?

**Parte C — LightGBM tuneado con Optuna (25 min)**
8. Definir un objective con los 9 hiperparametros principales, scoring='roc_auc'.
9. Correr 30 trials.
10. Reportar mejores hiperparametros + metricas en test.
11. Graficar la importancia de hiperparametros con `optuna.importance.get_param_importances(study)`.
12. Graficar la historia de optimizacion (valor por trial).

**Parte D — Analisis del ganador (15 min)**
13. Comparar los 4 modelos en una tabla final (Accuracy, F1, AUC-ROC, tiempo).
14. Graficar Feature Importance del LightGBM tuneado (con `importance_type='gain'`).
15. Mostrar la matriz de confusion del modelo ganador.
16. Cual gano? La mejora justifica el tiempo extra del tuning?
17. Que variables explican mejor si una persona gana mas de 50K segun el modelo? Tienen sentido?

**Parte E — Decision final**
18. Imagina que esta es una herramienta para que un banco prescore clientes para creditos premium. Cual modelo entregarian a produccion? Justifiquen pensando en: AUC, velocidad, explicabilidad y sesgo (el dataset tiene sesgo demografico, hay que discutirlo).

### Bonus que vale puntos extra

- Probar LightGBM con `boosting_type='dart'`: cambia el rendimiento?
- Subir n_trials de Optuna a 100: cuanto mejora el AUC? Vale la pena el tiempo?
- Combinar early stopping con Optuna: en el objective, usar fit con eval_set y early_stopping callback.
- Probar `is_unbalanced=True` o `scale_pos_weight` para manejar el desbalance de clases: cambian las metricas?

In [ ]:
# Parte A — 3 baselines



In [ ]:
# Parte B — LightGBM con early stopping



In [ ]:
# Parte C — LightGBM tuneado con Optuna



In [ ]:
# Parte D — Analisis del ganador



In [ ]:
# Parte E — Decision final



## Cierre

Hoy aprendimos:

- La diferencia entre Bagging (paralelo, RF) y Boosting (secuencial, LightGBM).
- Por que LightGBM es 10-20x mas rapido que XGBoost (histogram-based + leaf-wise).
- Los hiperparametros que realmente importan (n_estimators, learning_rate, num_leaves, etc).
- Early stopping para evitar overfitting y ahorrar tiempo.
- Como combinar LightGBM con Optuna para un tuning profesional.

Manana cerramos con XGBoost, que es el hermano mas antiguo y aun relevante de LightGBM.